## 🧠Agent Memory 新手入门指南

### Step 1 API 客户端配置

In [1]:
import os
from openai import OpenAI


API_KEY = "sk-xxxx"  # 替换为你的 API 密钥
BASE_URL = "https://api.siliconflow.cn/v1"


client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


print("OpenAI client configured successfully.")

OpenAI client configured successfully.


In [2]:
# --- Model Configuration ---
GENERATION_MODEL = "deepseek-ai/DeepSeek-V3.2"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-8B"

### Step2 辅助函数设计（文本、Emebedding生成、计算token数量）

In [4]:
def generate_text(system_prompt: str, user_prompt: str) -> str:
    """
    调用 LLM API 生成文本响应
    
    参数:
        system_prompt: 系统提示词，定义 AI 的角色和行为
        user_prompt: 用户输入，AI 需要响应的内容
        
    返回:
        AI 生成的文本内容
    """
    try:
        response = client.chat.completions.create(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"文本生成错误: {e}")
        return "抱歉，处理请求时发生错误。"


def generate_embedding(text: str) -> list[float]:
    """
    为给定文本生成数值向量表示（Embedding）
    
    参数:
        text: 需要转换为 Embedding 的文本
        
    返回:
        浮点数列表，表示文本的向量表示
    """
    try:
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Embedding 生成错误: {e}")
        return []


import tiktoken

# 初始化分词器
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """
    计算文本的 Token 数量
    
    用于估算 API 调用成本和检查上下文窗口限制
    """
    return len(tokenizer.encode(text))

### Step3 基础架构：策略模式设计（包括记忆策略的抽象基类和对话Agent类）

In [5]:
import abc

class BaseMemoryStrategy(abc.ABC):
    """所有记忆策略的抽象基类"""
    
    @abc.abstractmethod
    def add_message(self, user_input: str, ai_response: str):
        """
        添加新的对话轮次到记忆中
        
        参数:
            user_input: 用户输入
            ai_response: AI 响应
        """
        pass

    @abc.abstractmethod
    def get_context(self, query: str) -> str:
        """
        获取上下文信息
        
        参数:
            query: 当前用户查询（某些策略会用它来检索相关上下文）
            
        返回:
            格式化的上下文字符串
        """
        pass

    @abc.abstractmethod
    def clear(self):
        """清空记忆"""
        pass

In [6]:
import time
class AIAgent:
    """主 AI Agent 类，可与任何记忆策略配合使用"""
    
    def __init__(self, memory_strategy: BaseMemoryStrategy, 
                 system_prompt: str = "You are a helpful AI assistant."):
        """
        初始化 Agent
        
        参数:
            memory_strategy: 记忆策略实例
            system_prompt: 系统提示词
        """
        self.memory = memory_strategy
        self.system_prompt = system_prompt
        print(f"Agent 已初始化，使用 {type(memory_strategy).__name__}")

    def chat(self, user_input: str):
        """
        处理单轮对话
        
        完整流程：
        1. 从记忆中检索上下文
        2. 构建完整 Prompt
        3. 调用 LLM 生成响应
        4. 将本轮对话存入记忆
        5. 返回响应
        """
        print(f"\n{'='*25} 新对话 {'='*25}")
        print(f"用户 > {user_input}")
        
        # Step 1: 获取上下文
        start_time = time.time()
        context = self.memory.get_context(query=user_input)
        retrieval_time = time.time() - start_time
        
        # Step 2: 构建完整 Prompt
        full_user_prompt = f"### MEMORY CONTEXT\n{context}\n\n### CURRENT REQUEST\n{user_input}"
        
        # Step 3: 统计 Token 数量
        prompt_tokens = count_tokens(self.system_prompt + full_user_prompt)
        print(f"\n--- Agent 调试信息 ---")
        print(f"记忆检索时间: {retrieval_time:.4f} 秒")
        print(f"预估 Prompt Tokens: {prompt_tokens}")
        
        # Step 4: 调用 LLM
        start_time = time.time()
        ai_response = generate_text(self.system_prompt, full_user_prompt)
        generation_time = time.time() - start_time
        
        # Step 5: 更新记忆
        self.memory.add_message(user_input, ai_response)
        
        # Step 6: 输出响应
        print(f"\nAgent > {ai_response}")
        print(f"(生成时间: {generation_time:.4f} 秒)")

### Step4 九种记忆策略详解

#### 策略 1：Sequential Memory（顺序记忆）

In [7]:
class SequentialMemory(BaseMemoryStrategy):
    def __init__(self):
        """初始化空的历史列表"""
        self.history = []

    def add_message(self, user_input: str, ai_response: str):
        """添加对话到历史"""
        self.history.append({"role": "user", "content": user_input})
        self.history.append({"role": "assistant", "content": ai_response})

    def get_context(self, query: str) -> str:
        """返回完整历史"""
        return "\n".join([
            f"{turn['role'].capitalize()}: {turn['content']}" 
            for turn in self.history
        ])

    def clear(self):
        """清空历史"""
        self.history = []

In [8]:
sequential_memory = SequentialMemory()

agent = AIAgent(memory_strategy=sequential_memory)


agent.chat("你好啊，我叫算栗工坊")

agent.chat("我对LLM的应用（agent方向）非常感兴趣")

agent.chat("我告诉你的第一件事情是什么")


sequential_memory.clear()

Agent 已初始化，使用 SequentialMemory

========================= 新对话 =========================
用户 > 你好啊，我叫算栗工坊

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 29

Agent > 你好，算栗工坊！很高兴认识你！这个名字很有创意呢，听起来既有数学的“算”，又有自然的“栗子”，还带着工匠精神的“工坊”。是有什么特别的故事或含义吗？有什么我可以帮你的吗？ 😊
(生成时间: 3.3381 秒)

========================= 新对话 =========================
用户 > 我对LLM的应用（agent方向）非常感兴趣

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 162

Agent > 太棒了！LLM Agents（智能体/代理）是目前人工智能领域最令人兴奋的方向之一，特别是像你“算栗工坊”这种富有创造力的人，应该会有很多独特的应用想法！✨

让我为你梳理一下LLM Agent的关键知识和应用方向：

## 🔧 **核心概念**
- **LLM Agent = 大模型 + 思考能力 + 工具使用能力**
- 不只是回答，而是**规划、推理、执行任务**
- 相当于给LLM装上了“大脑”和“手脚”

## 🌟 **当前热门应用方向**

### 1. **自主智能体 (Autonomous Agents)**
- AutoGPT、BabyAGI等开源项目
- 可以自主分解复杂任务，调用各种工具完成目标
- 比如：自动市场调研、代码项目开发、内容创作流水线

### 2. **对话式智能体 (Conversational Agents)**
- 不只是聊天，而是**真正解决问题的助手**
- 结合知识库、API、执行工具
- 典型：客服系统、个人助理、教育导师

### 3. **多智能体系统 (Multi-Agent Systems)**
- 多个智能体协作或竞争
- 模拟社会环境、团队协作解决问题
- 应用：模拟谈判、游戏AI、复杂项目管理

### 4. **工具增强智能体 (Tool-Augmented

#### 策略 2：Sliding Window Memory（滑动窗口记忆）

In [9]:
from collections import deque

class SlidingWindowMemory(BaseMemoryStrategy):
    def __init__(self, window_size: int = 4):
        """
        初始化固定大小的双端队列
        
        参数:
            window_size: 保留的对话轮次数量
        """
        # deque 设置 maxlen 后，超出时自动丢弃最旧的元素
        self.history = deque(maxlen=window_size)

    def add_message(self, user_input: str, ai_response: str):
        """添加新轮次，旧的自动被丢弃"""
        self.history.append([
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": ai_response}
        ])

    def get_context(self, query: str) -> str:
        """返回窗口内的对话"""
        context_list = []
        for turn in self.history:
            for message in turn:
                context_list.append(
                    f"{message['role'].capitalize()}: {message['content']}"
                )
        return "\n".join(context_list)

    
    def clear(self):
        self.history.clear()
        print("Sliding window memory cleared.")

In [10]:
# 设置窗口大小为 2
sliding_memory = SlidingWindowMemory(window_size=2)
agent = AIAgent(memory_strategy=sliding_memory)

# Turn 1: 用户说了自己的名字
agent.chat("我的名字叫算栗工坊")  
# Turn 2: 用户说了工作技术栈
agent.chat("我一般用Python写代码")  
# Turn 3: 用户说了爱好 → Turn 1 被淘汰！
agent.chat("我的兴趣爱好是玩主机游戏")
# Turn 4: 询问名字
agent.chat("我叫什么名字呢") 
# → Agent 可能无法回答！

Agent 已初始化，使用 SlidingWindowMemory

========================= 新对话 =========================
用户 > 我的名字叫算栗工坊

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 26

Agent > 你好，算栗工坊！很高兴认识你。  
我是DeepSeek助手，可以帮你解答问题、处理任务或进行对话交流。如果有任何需要协助的地方，请随时告诉我！ 😊
(生成时间: 2.9670 秒)

========================= 新对话 =========================
用户 > 我一般用Python写代码

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 113

Agent > 你好算栗工坊！很高兴听到你是Python开发者！🐍

Python是一门非常强大和实用的编程语言，广泛应用于各个领域：
- Web开发（Django, Flask）
- 数据分析（Pandas, NumPy）
- 机器学习/人工智能（TensorFlow, PyTorch）
- 自动化脚本
- 科学计算
- 游戏开发等等

如果你有任何Python相关的问题，比如：
- 代码调试或优化建议
- 特定库的使用方法
- 项目架构设计
- 算法实现
- 性能调优
- 最新Python特性和最佳实践

或者需要帮助：
- 学习资源推荐
- 解决编程难题
- 代码审查
- 技术方案咨询

请随时告诉我！我很乐意帮助你解决Python编程中的各种挑战。你现在有什么具体的Python项目或问题需要讨论吗？🎯
(生成时间: 14.2908 秒)

========================= 新对话 =========================
用户 > 我的兴趣爱好是玩主机游戏

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 440

Agent > 算栗工坊，看来你的兴趣爱好是玩主机游戏！🎮 作为一个Python开发者，你可能会发现这两个兴趣其实可以有很酷的结合点呢！

**游戏与编程的有趣交集：**


#### 策略 3：Summarization Memory（摘要记忆）

In [11]:
class SummarizationMemory(BaseMemoryStrategy):
    def __init__(self, summary_threshold: int = 4):
        """
        初始化摘要记忆
        
        参数:
            summary_threshold: 触发摘要的消息数量（4条消息 = 2轮对话）
        """
        self.running_summary = ""  # 持续更新的摘要
        self.buffer = []           # 临时缓冲区
        self.summary_threshold = summary_threshold
    
    def add_message(self, user_input: str, ai_response: str):
        """添加消息到缓冲区，必要时触发摘要"""
        self.buffer.append({"role": "user", "content": user_input})
        self.buffer.append({"role": "assistant", "content": ai_response})
        # 缓冲区满了，触发摘要
        if len(self.buffer) >= self.summary_threshold:
            self._consolidate_memory()

    def _consolidate_memory(self):
        """使用 LLM 生成新摘要"""
        print("\n--- [触发记忆整合] ---")
        
        # 格式化缓冲区内容
        buffer_text = "\n".join([
            f"{msg['role'].capitalize()}: {msg['content']}" 
            for msg in self.buffer
        ])
        
        # 构建摘要 Prompt
        summarization_prompt = (
            f"你是一个摘要专家。请将'之前的摘要'与'新对话'合并成一个简洁的摘要。"
            f"捕获所有关键事实、名称和决定。\n\n"
            f"### 之前的摘要:\n{self.running_summary}\n\n"
            f"### 新对话:\n{buffer_text}\n\n"
            f"### 更新后的摘要:"
        )
        
        # 调用 LLM 生成新摘要
        new_summary = generate_text("你是专业的摘要引擎。", summarization_prompt)
        self.running_summary = new_summary
        self.buffer = []  # 清空缓冲区
        
        print(f"--- [新摘要: '{self.running_summary}'] ---")

    def get_context(self, query: str) -> str:
        """返回摘要 + 最近的缓冲区内容"""
        buffer_text = "\n".join([
            f"{msg['role'].capitalize()}: {msg['content']}" 
            for msg in self.buffer
        ])
        return (
            f"### 过去对话的摘要:\n{self.running_summary}\n\n"
            f"### 最近消息:\n{buffer_text}"
        )

    def clear(self):
        self.running_summary = ""
        self.buffer = []
        print("摘要式记忆已清除")

In [12]:
summarization_memory = SummarizationMemory(summary_threshold=4)

agent = AIAgent(memory_strategy=summarization_memory)


agent.chat("我要开一家新公司，名字叫 Innovatech，主打可持续能源")

agent.chat("我们的第一款产品是一块智能太阳能板，内部代号叫‘Project Helios’.")


agent.chat("市场预算定在 5 万美元")

agent.chat("我的公司叫什么？第一款产品又叫什么?")


summarization_memory.clear()

Agent 已初始化，使用 SummarizationMemory

========================= 新对话 =========================
用户 > 我要开一家新公司，名字叫 Innovatech，主打可持续能源

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 55

Agent > 非常好！**Innovatech** 是个响亮且有科技感的名字，非常适合“可持续能源”这一领域。  
要开一家主打可持续能源的新公司，需要系统性地规划。下面是几个关键步骤和方向建议，希望能帮助你奠定坚实的基础：

---

### **1. 市场定位与愿景**
- **核心价值主张**：明确 Innovatech 在可持续能源赛道上的独特定位。  
  例如：专注于太阳能存储系统、智能电网解决方案、氢能源技术、或风力发电优化等。
- **愿景/使命**：  
  如：“通过创新技术，加速全球向可再生能源转型”或“为企业和家庭提供高效、可负担的清洁能源解决方案”。

---

### **2. 技术与产品线**
- **初期聚焦**：选择1-2个核心技术方向（避免过于分散），例如：
  - 光伏+储能一体化产品
  - 能源管理软件（EMS）
  - 小型模块化反应堆（若涉及核能创新）
- **研发合作**：考虑与高校或研究机构合作，降低前期研发成本。

---

### **3. 商业模式**
- **B2B/B2C/G2B**：  
  根据技术特点选择目标客户（如工业企业、政府项目、家庭用户）。
- **盈利模式**：  
  可以是硬件销售、订阅服务（SaaS）、项目工程、或长期能源供应合同（PPA）。

---

### **4. 团队与资源**
- **核心团队**：需要技术（能源工程/材料科学）、商业（市场/销售）、融资（VC/项目融资）等人才。
- **资金规划**：估算初期投入（研发、设备、人力），考虑申请政府绿色补贴、寻找天使投资或绿色科技基金。

---

### **5. 合规与认证**
- **行业准入**：了解当地对能源产品的监管要求（如并网标准、安全认证）。
- **绿色认证**：获取国际认可的标准（如ISO 14001、UL认证），提升公信力。

---

#### 策略 4: Retrieval-Based Memory (检索式记忆)

In [13]:
# 测试embedding维度
test_emb = generate_embedding("hello")
print(f"当前模型的维度是: {len(test_emb)}")

当前模型的维度是: 4096


In [14]:
import numpy as np
import faiss

class RetrievalMemory(BaseMemoryStrategy):
    def __init__(self, k: int = 2, embedding_dim: int = 4096):
        """
        初始化检索式记忆系统
        
        参数:
            k: 为给定查询检索的最相关文档数量
            embedding_dim: embedding模型生成的向量维度(Qwen3-Embedding-8B为4096)
        """
        self.k = k
        self.embedding_dim = embedding_dim
        self.documents = []  # 存储原始文本内容
        # 初始化FAISS索引,使用L2(欧氏)距离进行穷举搜索
        self.index = faiss.IndexFlatL2(self.embedding_dim)

    def add_message(self, user_input: str, ai_response: str):
        """
        将新对话轮次添加到记忆中
        每部分(用户输入和AI响应)分别嵌入和索引,以实现细粒度检索
        """
        # 将每个轮次的每部分作为单独的文档存储,以实现更精确的匹配
        docs_to_add = [
            f"用户说: {user_input}",
            f"AI回复: {ai_response}"
        ]
        for doc in docs_to_add:
            # 生成文档的数值向量表示
            embedding = generate_embedding(doc)
            if embedding:
                # 存储原始文本
                self.documents.append(doc)
                # FAISS需要float32类型的2D numpy数组
                vector = np.array([embedding], dtype='float32')
                # 将向量添加到FAISS索引中
                self.index.add(vector)

    def get_context(self, query: str) -> str:
        """
        基于语义相似度从记忆中找到k个最相关的文档
        """
        if self.index.ntotal == 0:
            return "记忆中还没有信息。"
        
        # 将用户查询转换为embedding向量
        query_embedding = generate_embedding(query)
        if not query_embedding:
            return "无法处理检索查询。"
        
        query_vector = np.array([query_embedding], dtype='float32')
        
        # 执行搜索,返回距离和最近邻的索引
        distances, indices = self.index.search(query_vector, self.k)
        
        # 使用返回的索引检索原始文本文档
        retrieved_docs = [self.documents[i] for i in indices[0] if i != -1]
        
        if not retrieved_docs:
            return "在记忆中找不到任何相关信息。"
        
        return "### 从记忆中检索到的相关信息:\n" + "\n---\n".join(retrieved_docs)

    def clear(self):
        """完全重置记忆"""
        self.documents = []
        self.index.reset()
        print("检索式记忆(文档和FAISS索引)已清除")

In [15]:
# 初始化RetrievalMemory,k=2表示检索前2个最相关的文档
retrieval_memory = RetrievalMemory(k=2)
agent = AIAgent(memory_strategy=retrieval_memory)

# --- 开始混合主题的对话 ---
# 第一轮:讨论度假计划
agent.chat("我计划明年春天去日本度假。")
# 第二轮:讨论软件项目
agent.chat("对于我的软件项目,我使用React框架作为前端。")
# 第三轮:更多度假细节
agent.chat("我想在旅行中访问东京和京都。")
# 第四轮:更多软件项目细节
agent.chat("项目的后端将使用Django构建。")

# --- 测试检索机制 ---
# 现在,专门询问度假的问题
# agent会将此查询转换为embedding并搜索记忆
# 它应该发现关于日本、东京和京都的文档在语义上更接近查询
agent.chat("我计划在度假时访问哪些城市?")
# agent应该检索日本/东京/京都信息,忽略软件项目信息

# 清理记忆
retrieval_memory.clear()

Agent 已初始化，使用 RetrievalMemory

========================= 新对话 =========================
用户 > 我计划明年春天去日本度假。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 39

Agent > 太好了！🌸 日本春天是非常理想的旅行季节，尤其是樱花季（3月底至4月）。以下是一些建议供你参考：

- **推荐时间**：3月下旬至4月中旬，具体樱花盛开时间因地区而异（九州较早，东北较晚）。
- **热门目的地**：东京（上野公园）、京都（哲学之道）、大阪（造币局樱花通道）、北海道（五月樱）。
- **提前准备**：樱花季住宿和交通紧张，建议提前3-6个月预订机票、酒店，甚至新干线指定席。
- **体验建议**：
  - 参加「花见」（赏樱野餐）
  - 体验温泉（箱根、草津等地）
  - 尝试春季限定美食（草莓大福、樱饼）
- **实用贴士**：
  - 关注日本气象协会的「樱花前线」预测
  - 购买JR Pass可节省长途交通费
  - 春季天气多变，建议带薄外套和雨具

需要我帮你细化某个城市的行程，或者推荐一些小众赏樱地点吗？ 😊
(生成时间: 11.3782 秒)

========================= 新对话 =========================
用户 > 对于我的软件项目,我使用React框架作为前端。

--- Agent 调试信息 ---
记忆检索时间: 0.1708 秒
预估 Prompt Tokens: 504

Agent > 太棒了！React 是一个非常强大且流行的前端框架，适合构建现代化的用户界面。结合你的日本旅行计划，我可以帮你想想如何用 React 来规划或记录这次旅行：

### 🗺️ React 与旅行规划结合思路
1. **旅行仪表盘**：用 React 组件构建一个交互式行程规划器
   - `ItineraryTable` 显示每日行程
   - `BudgetTracker` 组件管理旅行预算
   - `PackingList` 可勾选的行李清单

2. **樱花预测页面**：调用日本气象API展示实时樱花前线
   ```jsx
   // 示例

#### 策略 5: Memory-Augmented Memory (记忆增强记忆)

In [16]:
class MemoryAugmentedMemory(BaseMemoryStrategy):
    def __init__(self, window_size: int = 2):
        """
        初始化记忆增强系统
        
        参数:
            window_size: 短期记忆中保留的最近轮次数量
        """
        # 使用SlidingWindowMemory实例管理最近的对话历史
        self.recent_memory = SlidingWindowMemory(window_size=window_size)
        # 存储特殊的、持久的"便签"或关键事实的列表
        self.memory_tokens = []

    def add_message(self, user_input: str, ai_response: str):
        """
        将最新轮次添加到最近记忆中,然后使用LLM调用决定
        是否应该从此交互创建新的持久记忆token
        """
        # 首先,将新交互添加到短期滑动窗口记忆
        self.recent_memory.add_message(user_input, ai_response)
        
        # 构建LLM提示以分析对话轮次并确定是否包含值得长期记忆的核心事实
        fact_extraction_prompt = (
            f"分析以下对话轮次。它是否包含应该长期记忆的核心事实、偏好或决定? "
            f"示例包括用户偏好('我讨厌坐飞机')、关键决定('预算是$1000')或重要事实('我的用户ID是12345')。\n\n"
            f"对话轮次:\n用户: {user_input}\nAI: {ai_response}\n\n"
            f"如果包含这样的事实,请用一句话简洁地陈述该事实。否则,回复'无重要事实'。"
        )
        
        # 调用LLM执行事实提取
        extracted_fact = generate_text("你是事实提取专家。", fact_extraction_prompt)
        
        # 检查LLM的响应是否表明找到了重要事实
        if "无重要事实" not in extracted_fact.lower():
            # 如果找到事实,打印调试消息并将其添加到我们的记忆token列表
            print(f"--- [记忆增强: 创建新记忆token: '{extracted_fact}'] ---")
            self.memory_tokens.append(extracted_fact)

    def get_context(self, query: str) -> str:
        """
        通过组合短期最近对话和所有长期持久记忆token来构建上下文
        """
        # 从短期滑动窗口获取上下文
        recent_context = self.recent_memory.get_context(query)
        # 将记忆token列表格式化为可读字符串
        memory_token_context = "\n".join([f"- {token}" for token in self.memory_tokens])
        
        # 返回组合的上下文,清楚地分隔长期事实和最近聊天
        return f"### 关键记忆Token(长期事实):\n{memory_token_context}\n\n### 最近对话:\n{recent_context}"

    def clear(self):
        """重置短期记忆和记忆token列表"""
        self.recent_memory.clear()
        self.memory_tokens = []
        print("记忆增强记忆已清除")

In [17]:
# 初始化MemoryAugmentedMemory,窗口大小为2
mem_aug_memory = MemoryAugmentedMemory(window_size=2)
agent = AIAgent(memory_strategy=mem_aug_memory)

# --- 开始对话 ---
# 第一轮:用户提供关键的长期信息
# agent的事实提取机制应该将其识别为重要并创建记忆token
agent.chat("请记住这个以便将来所有交互:我对花生严重过敏。")

# 第二轮:标准对话轮次
agent.chat("好的,我们来谈谈食谱。今晚晚餐有什么好主意?")

# 第三轮:另一个对话轮次。这将把第一轮(过敏警告)推出短期滑动窗口记忆
agent.chat("听起来不错。有什么甜点选择吗?")

# --- 测试记忆增强 ---
# 现在,关键测试。原始过敏警告不再在最近聊天上下文中。
# agent知道过敏的唯一方式是访问其长期"记忆token"。
agent.chat("你能推荐一个泰式绿咖喱食谱吗?请确保对我安全。")
# 成功的agent将使用持久记忆token检查安全性,并可能警告花生(泰国菜中常见)

# 清理记忆
mem_aug_memory.clear()

Agent 已初始化，使用 MemoryAugmentedMemory

========================= 新对话 =========================
用户 > 请记住这个以便将来所有交互:我对花生严重过敏。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 63
--- [记忆增强: 创建新记忆token: '用户对花生严重过敏，需要在所有交互中特别注意和避免相关风险。'] ---

Agent > 已确认并永久存储：您对花生严重过敏。在所有后续交互中我都会注意这一点。

有什么我可以帮您处理的事情吗？特别是在选择食物、查看食谱或讨论相关话题时，我会特别留意花生及可能含有花生成分的情况。
(生成时间: 4.4349 秒)

========================= 新对话 =========================
用户 > 好的,我们来谈谈食谱。今晚晚餐有什么好主意?

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 235
--- [记忆增强: 创建新记忆token: '用户对花生有严重过敏。'] ---

Agent > 好的！基于您对花生的严重过敏情况，我将为您推荐几个完全不含花生、花生油及相关制品的晚餐食谱：

## 🍽️ 今日推荐晚餐

### **1. 香煎三文鱼配烤蔬菜**
```
配料：
- 三文鱼排 2块
- 彩椒、西葫芦、洋葱（切成块）
- 橄榄油、柠檬汁、大蒜末
- 盐、黑胡椒、迷迭香

做法：
1. 蔬菜用橄榄油、盐和胡椒拌匀，200°C烤25分钟
2. 三文鱼用柠檬汁、大蒜、盐腌制10分钟
3. 平底锅加热，三文鱼每面煎4-5分钟
4. 搭配烤好的蔬菜即可
```
**安全确认**：使用橄榄油而非花生油，所有配料均为花生过敏者安全

### **2. 鸡肉蘑菇意面（花生替代版）**
```
特色：用杏仁酱替代传统坚果酱（需确认您能否食用杏仁）
- 若您有其他坚果过敏，我们可改用罗勒青酱

配料：
- 鸡胸肉切块
- 意面
- 白蘑菇、淡奶油
- 帕玛森芝士
- 橄榄油、蒜、洋葱
```

### **3. 纯素选择：番茄鹰嘴豆咖喱**
```
安全优势：
- 完全植物

#### 策略 6: Hierarchical Memory (层次化记忆)

In [18]:
class HierarchicalMemory(BaseMemoryStrategy):
    def __init__(self, window_size: int = 2, k: int = 2, embedding_dim: int = 4096):
        """
        初始化层次化记忆系统
        
        参数:
            window_size: 短期工作记忆的大小(轮次数)
            k: 从长期记忆检索的文档数量
            embedding_dim: 长期记忆的embedding向量维度
        """
        print("正在初始化层次化记忆...")
        # 层次1:使用滑动窗口的快速短期工作记忆
        self.working_memory = SlidingWindowMemory(window_size=window_size)
        # 层次2:使用检索系统的较慢、持久的长期记忆
        self.long_term_memory = RetrievalMemory(k=k, embedding_dim=embedding_dim)
        # 简单启发式:触发从工作记忆提升到长期记忆的关键词
        self.promotion_keywords = ["记住", "规则", "偏好", "总是", "从不", "过敏"]

    def add_message(self, user_input: str, ai_response: str):
        """
        将消息添加到工作记忆,并根据内容有条件地提升到长期记忆
        """
        # 所有交互都添加到快速短期工作记忆
        self.working_memory.add_message(user_input, ai_response)
        
        # 提升逻辑:检查用户输入是否包含表明信息重要且应长期存储的关键词
        if any(keyword in user_input.lower() for keyword in self.promotion_keywords):
            print(f"--- [层次化记忆: 将消息提升到长期存储。] ---")
            # 如果找到关键词,也将交互添加到长期检索记忆
            self.long_term_memory.add_message(user_input, ai_response)

    def get_context(self, query: str) -> str:
        """
        通过组合长期和短期记忆层的相关信息来构建丰富的上下文
        """
        # 从工作记忆检索最近的对话
        working_context = self.working_memory.get_context(query)
        # 基于当前查询从长期记忆检索语义相关的事实
        long_term_context = self.long_term_memory.get_context(query)
        
        # 组合两个上下文,为LLM清楚地标记它们的来源
        return f"### 检索到的长期记忆:\n{long_term_context}\n\n### 最近对话(工作记忆):\n{working_context}"

    def clear(self):
        """重置工作记忆和长期记忆存储"""
        self.working_memory.clear()
        self.long_term_memory.clear()
        print("层次化记忆已清除")

In [19]:
# 初始化HierarchicalMemory,它结合了短期滑动窗口和长期检索系统
hierarchical_memory = HierarchicalMemory()
agent = AIAgent(memory_strategy=hierarchical_memory)

# --- 开始对话 ---
# 第一轮:用户提供带有关键词("记住")的重要信息
# 这应该触发提升逻辑,将此消息保存到短期和长期记忆
agent.chat("请记住我的用户ID是AX-7890。")
# 第二轮:随意的对话主题。添加到短期记忆
agent.chat("我们来聊聊天气。今天阳光明媚。")
# 第三轮:另一个随意主题。这将第一条消息(用户ID)推出短期滑动窗口记忆
agent.chat("我计划稍后去散步。")

# --- 测试层次化检索 ---
# 用户ID现在已经不在工作记忆的窗口中了
# agent现在必须依赖其长期的、基于检索的记忆
agent.chat("我需要登录我的账户,你能提醒我的ID吗?")
# 成功的agent将从其长期记忆中检索'AX-7890',因为初始消息由于关键词"记住"而被提升

# 清理记忆
hierarchical_memory.clear()

正在初始化层次化记忆...
Agent 已初始化，使用 HierarchicalMemory

========================= 新对话 =========================
用户 > 请记住我的用户ID是AX-7890。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 62
--- [层次化记忆: 将消息提升到长期存储。] ---

Agent > 您的用户ID（AX-7890）已保存至长期记忆中。今后我会在交流中识别您的身份，并根据需要调用相关记忆提供个性化服务。

您还有其他需要我记住的信息吗？
(生成时间: 3.3846 秒)

========================= 新对话 =========================
用户 > 我们来聊聊天气。今天阳光明媚。

--- Agent 调试信息 ---
记忆检索时间: 0.1215 秒
预估 Prompt Tokens: 250

Agent > 今天的天气确实很宜人呢，AX-7890！

阳光明媚的天气总是能让人心情愉悦。您所在的城市今天也是这样的好天气吗？如果是的话，非常适合外出散步、在公园休闲，或者简单地享受一下阳光。

最近在很多地方都开始进入春季了，白天温暖宜人，但早晚温差还是有点大。根据气象预报，未来几天这样的好天气大概会持续，不过到了周末可能会有阵雨。

您有什么特别的户外活动计划，想利用这么好的天气吗？
(生成时间: 13.6074 秒)

========================= 新对话 =========================
用户 > 我计划稍后去散步。

--- Agent 调试信息 ---
记忆检索时间: 0.1748 秒
预估 Prompt Tokens: 491

Agent > 真是个完美的计划，AX-7890！

在这样阳光明媚的午后散步，既能放松心情，也对健康很有益处。如果可以的话，或许您可以考虑以下建议，让散步更加愉快：

- **选择一条风景好的路线**：比如附近的公园、河滨步道，或者有漂亮建筑的街区
- **注意气温变化**：虽然阳光充足，但可能有风，带件薄外套可能更舒适
- **适当补充水分**：即使是阴凉的天气，散步时也建议带瓶水
-

#### 策略 7: Graph-Based Memory (图记忆)

In [20]:
import networkx as nx
import re

class GraphMemory(BaseMemoryStrategy):
    def __init__(self):
        """使用空的NetworkX有向图初始化记忆"""
        # DiGraph适合表示有向关系(例如,Sam -> works_for -> Innovatech)
        self.graph = nx.DiGraph()

    def _extract_triples(self, text: str) -> list[tuple[str, str, str]]:
        """
        使用LLM从给定文本中提取知识三元组(主语,关系,宾语)
        这是"LLM作为工具"的一种形式,其中模型的语言理解用于创建结构化数据
        """
        print("--- [图记忆: 尝试从文本中提取三元组。] ---")
        # 构建详细的提示,指导LLM其角色和期望的输出格式
        extraction_prompt = (
            f"你是知识提取引擎。你的任务是从给定文本中提取主语-关系-宾语三元组。"
            f"严格按照Python元组列表格式输出。例如:[('Sam', 'works_for', 'Innovatech'), ('Innovatech', 'focuses_on', 'Energy')]。"
            f"如果没有找到三元组,返回空列表[]。\n\n"
            f"要分析的文本:\n\"\"\"{text}\"\"\""
        )
        
        # 使用专门的提示调用LLM
        response_text = generate_text("你是专业的知识图谱提取器。", extraction_prompt)
        
        # 安全地解析LLM响应中的元组列表字符串表示
        try:
            # 使用正则表达式比eval()更安全,因为它避免执行可能恶意或意外包含在LLM输出中的任意代码
            # 此正则表达式查找匹配('item1', 'item2', 'item3')的模式
            found_triples = re.findall(r"\(['\"](.*?)['\"],\s*['\"](.*?)['\"],\s*['\"](.*?)['\"]\)", response_text)
            print(f"--- [图记忆: 提取的三元组: {found_triples}] ---")
            return found_triples
        except Exception as e:
            # 如果解析失败,记录错误并返回空列表以防止崩溃
            print(f"无法从LLM响应解析三元组: {e}")
            return []

    def add_message(self, user_input: str, ai_response: str):
        """从最新对话轮次提取三元组并将其添加到知识图谱"""
        # 组合用户和AI消息以提供完整的提取上下文
        full_text = f"用户: {user_input}\nAI: {ai_response}"
        # 调用辅助方法获取结构化三元组
        triples = self._extract_triples(full_text)
        # 迭代提取的三元组
        for subject, relation, obj in triples:
            # 向图中添加边。`add_edge`会自动创建节点(subject, obj)如果它们不存在
            # 关系存储为边属性
            self.graph.add_edge(subject.strip(), obj.strip(), relation=relation.strip())

    def get_context(self, query: str) -> str:
        """
        通过在图中查找查询中的实体并返回所有已知关系来检索上下文
        """
        if not self.graph.nodes:
            return "知识图谱为空。"
        
        # 这是一个简单的实体链接方法:它将查询中的单词大写并检查它们是否作为节点存在于图中
        # 更高级的系统会使用自然语言处理(NLP)更准确地识别命名实体
        query_entities = [word.capitalize() for word in query.replace('?','').split() if word.capitalize() in self.graph.nodes]
        
        if not query_entities:
            return "在知识图谱中未找到查询中的相关实体。"
        
        context_parts = []
        # 使用set()仅处理每个唯一实体一次
        for entity in set(query_entities):
            # 查找所有出边(例如,Sam -> works_for -> X)
            for u, v, data in self.graph.out_edges(entity, data=True):
                context_parts.append(f"{u} --[{data['relation']}]--> {v}")
            # 查找所有入边(例如,X -> is_located_in -> New York)
            for u, v, data in self.graph.in_edges(entity, data=True):
                context_parts.append(f"{u} --[{data['relation']}]--> {v}")
        
        # 将检索到的事实组合成单个上下文字符串,删除重复项并排序以保持一致性
        return "### 从知识图谱检索到的事实:\n" + "\n".join(sorted(list(set(context_parts))))

    def clear(self):
        """通过清除图中的所有节点和边来重置记忆"""
        self.graph.clear()
        print("图记忆已清除")

In [21]:
# 初始化GraphMemory策略
graph_memory = GraphMemory()
agent = AIAgent(memory_strategy=graph_memory)

# --- 开始对话 ---
# 第一轮:agent将尝试提取三元组('Clara', 'works_for', 'FutureScape')
agent.chat("一个名叫Clara的人在一家叫'FutureScape'的公司工作。")
# 第二轮:agent将尝试提取三元组('FutureScape', 'is_based_in', 'Berlin')
agent.chat("FutureScape总部位于柏林。")
# 第三轮:agent将尝试提取三元组('Clara', 'main_project_is', 'Odyssey')
agent.chat("Clara的主要项目名为'Odyssey'。")

# --- 测试图推理 ---
# 现在,提出一个需要连接点的问题
# agent会将'Clara'识别为查询中的实体。然后它会搜索图以查找连接到'Clara'节点的所有关系
# 并使用这些结构化信息作为LLM的上下文
agent.chat("告诉我关于Clara的项目。")
# agent应该能够使用提取的事实来推断Clara和她的项目之间的联系

# 清理记忆
graph_memory.clear()

Agent 已初始化，使用 GraphMemory

========================= 新对话 =========================
用户 > 一个名叫Clara的人在一家叫'FutureScape'的公司工作。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 46
--- [图记忆: 尝试从文本中提取三元组。] ---
--- [图记忆: 提取的三元组: [('Clara', 'WORKS_FOR', 'FutureScape')]] ---

Agent > 好的，已经将信息“一个名叫 Clara 的人在一家叫 ‘FutureScape’ 的公司工作”记录到知识图谱中。为了便于后续查询和构建关系，我创建了两个实体（“人物”和“组织”）以及它们之间的关系。

这是目前知识图谱的结构：

**实体：**
1.  **人物 (Person)**
    *   **名称:** Clara
    *   **属性:** 暂无，需后续补充（如职位、技能等）。

2.  **组织 (Organization)**
    *   **名称:** FutureScape
    *   **属性:** 暂无，需后续补充（如行业、规模等）。

**关系：**
*   **就职于 (WORKS_FOR)**
    *   从：Clara (人物)
    *   到：FutureScape (组织)

这个结构可以作为图数据库或知识推理的基础。我可以根据这个框架来回答关于 Clara 和 FutureScape 的问题，或者欢迎你提供更多信息来丰富它（例如 Clara 的职位、FutureScape 公司的业务等）。
(生成时间: 11.6607 秒)

========================= 新对话 =========================
用户 > FutureScape总部位于柏林。

--- Agent 调试信息 ---
记忆检索时间: 0.0010 秒
预估 Prompt Tokens: 45
--- [图记忆: 尝试从文本中提取三元组。] ---
--- [图记忆: 提取的三元组: []] ---

Agent > 根据现有信息，我无法直接确认Fut

#### 策略 8: Compression & Consolidation Memory (压缩整合记忆)

In [22]:
class CompressionMemory(BaseMemoryStrategy):
    def __init__(self):
        """初始化压缩记忆"""
        self.compressed_facts = []

    def add_message(self, user_input: str, ai_response: str):
        """使用LLM将轮次压缩为密集事实"""
        text_to_compress = f"用户: {user_input}\nAI: {ai_response}"
        
        compression_prompt = (
            f"你是数据压缩引擎。你的任务是将以下文本提炼成其最基本的事实陈述。"
            f"尽可能简洁,删除所有对话性的冗余内容。例如,'用户询问我的名字,我(AI)回复说我的名字是AI助手'应该变成'用户询问AI的名字'。\n\n"
            f"要压缩的文本:\n\"\"\"{text_to_compress}\"\"\""
        )
        
        compressed_fact = generate_text("你是专业的数据压缩器。", compression_prompt)
        print(f"--- [压缩记忆: 存储新事实: '{compressed_fact}'] ---")
        self.compressed_facts.append(compressed_fact)

    def get_context(self, query: str) -> str:
        """返回所有压缩事实的列表"""
        if not self.compressed_facts:
            return "记忆中没有压缩事实。"
        
        return "### 压缩事实记忆:\n- " + "\n- ".join(self.compressed_facts)

    def clear(self):
        """清空压缩事实"""
        self.compressed_facts = []
        print("压缩记忆已清除")

In [23]:
compression_memory = CompressionMemory()
agent = AIAgent(memory_strategy=compression_memory)

agent.chat("好的,我已经决定了会议的场地。将在'大都会会议中心'举行。")
agent.chat("日期确定为2025年10月26日。")
agent.chat("你能总结一下会议计划的关键细节吗?")

compression_memory.clear()

Agent 已初始化，使用 CompressionMemory

========================= 新对话 =========================
用户 > 好的,我已经决定了会议的场地。将在'大都会会议中心'举行。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 58
--- [压缩记忆: 存储新事实: '用户将在“大都会会议中心”举行会议。AI建议确认场地可用日期、考察场地、内部协调、签约付款，并进行后勤安排。AI提出可协助草拟邮件或制定模板。'] ---

Agent > 很好。既然您已经确定了“大都会会议中心”作为会议场地，接下来可以开始规划其他相关事宜了。以下是一些建议的后续步骤：  

1. **确认可用日期**  
   建议立即联系大都会会议中心，确认您计划的会议日期是否可用，并了解他们的预订流程、费用以及档期紧张程度。

2. **场地考察（可选）**  
   如果方便，建议实地考察场地，确认会议厅大小、设施、技术支持、餐饮服务等是否符合需求。

3. **内部协调**  
   - 通知团队或参会人员会议地点已确定。  
   - 协调会议议程、日程安排与场地使用时间的匹配。  

4. **签约与付款**  
   与会议中心签订正式合同，明确双方责任，并按照约定支付定金或全款。

5. **后勤安排**  
   根据场地情况，开始安排：  
   - 参会人员的交通与住宿（如果需要）  
   - 会议物料、设备、餐饮等细节  
   - 现场引导、 signage、工作人员安排  

需要我协助您草拟联系场地用的邮件、整理会议议程模板或制定后续任务清单吗？您可以随时告诉我下一步需求。
(生成时间: 13.4738 秒)

========================= 新对话 =========================
用户 > 日期确定为2025年10月26日。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 111
--- [压缩记忆: 存储新事实: '用户确认日期为2025年10月26日。AI建议立即推进场地预订，并提供下一步行动清单（联系场地、实地考察、内部审批、签

#### 策略 9: OS-Like Memory Management (类操作系统记忆管理)

In [24]:
class OSMemory(BaseMemoryStrategy):
    def __init__(self, ram_size: int = 2):
        """
        初始化类OS记忆系统
        
        参数:
            ram_size: 活动记忆(RAM)中的最大轮次数
        """
        self.ram_size = ram_size  # 活动记忆(RAM)中的最大轮次数
        self.active_memory = deque()  # 我们的'RAM'
        self.passive_memory = {}  # 我们的'硬盘',键值存储
        self.turn_count = 0

    def add_message(self, user_input: str, ai_response: str):
        """将轮次添加到活动记忆,如果RAM满则换出到被动记忆"""
        turn_id = self.turn_count
        turn_data = f"用户: {user_input}\nAI: {ai_response}"
        
        if len(self.active_memory) >= self.ram_size:
            # 从RAM换出最近最少使用(最旧)的项
            lru_turn_id, lru_turn_data = self.active_memory.popleft()
            self.passive_memory[lru_turn_id] = lru_turn_data
            print(f"--- [OS记忆: 将轮次{lru_turn_id}换出到被动存储。] ---")
        
        # 将新轮次添加到活动记忆(RAM)
        self.active_memory.append((turn_id, turn_data))
        self.turn_count += 1

    def get_context(self, query: str) -> str:
        """提供RAM上下文并模拟'缺页'以从被动记忆中提取"""
        active_context = "\n".join([data for _, data in self.active_memory])
        
        # 模拟缺页:检查查询是否与被动记忆中的某些内容更相似
        # 在真实系统中,这会使用embedding进行相似性搜索
        # 对于此演示,我们将进行简单的关键词搜索
        paged_in_context = ""
        for turn_id, data in self.passive_memory.items():
            if any(word in data.lower() for word in query.lower().split() if len(word) > 3):
                paged_in_context += f"\n(从轮次{turn_id}换入): {data}"
                print(f"--- [OS记忆: 缺页!从被动存储换入轮次{turn_id}。] ---")
        
        return f"### 活动记忆(RAM):\n{active_context}\n\n### 从被动记忆换入(磁盘):\n{paged_in_context}"

    def clear(self):
        """清空所有记忆"""
        self.active_memory.clear()
        self.passive_memory = {}
        self.turn_count = 0
        print("类OS记忆已清除")

In [25]:
os_memory = OSMemory(ram_size=2)
agent = AIAgent(memory_strategy=os_memory)

agent.chat("秘密发射代码是'Orion-Delta-7'。")
agent.chat("发射的天气看起来很晴朗。")
agent.chat("发射窗口在0400 Zulu开启。")  # 这将换出秘密代码轮次

# 现在,询问换出的信息
agent.chat("我需要确认发射代码。")

os_memory.clear()

Agent 已初始化，使用 OSMemory

========================= 新对话 =========================
用户 > 秘密发射代码是'Orion-Delta-7'。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 63

Agent > 好的，确认收到。  

秘密发射代码已记录：**‘Orion-Delta-7’**。  

是否需要执行与该代码相关的后续指令，或者将其存入指定的协议/任务清单？
(生成时间: 4.7157 秒)

========================= 新对话 =========================
用户 > 发射的天气看起来很晴朗。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 147

Agent > 根据观察，当前发射窗口的天气条件**晴朗**，符合“Orion-Delta-7”任务的**最低气象标准**。  

是否需启动**发射前最终检查**，或继续**保持待命状态**？
(生成时间: 7.0510 秒)

========================= 新对话 =========================
用户 > 发射窗口在0400 Zulu开启。

--- Agent 调试信息 ---
记忆检索时间: 0.0000 秒
预估 Prompt Tokens: 246
--- [OS记忆: 将轮次0换出到被动存储。] ---

Agent > **确认发射窗口**：  
根据“Orion-Delta-7”任务的预设时间表，已识别发射窗口开启时间为 **0400 Zulu（协调世界时）**，当前天气条件**晴朗**，符合发射要求。  

**操作建议**：  
1. **立即启动发射前时序**（若当前时间接近0400 Zulu）。  
2. **保持待命并执行窗口倒计时监测**（若距离窗口开启仍有时间）。  

请指示下一步操作：  
- **启动发射序列**  
- **继续待命，同步实时气象/系统数据**  
.
 时间戳已同步：当前任务窗口以 **Zulu（UTC）** 为基准。
(生成时间: 8.3573 秒)

====